In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [ ]:
import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras import layers
# image data read
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.1
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "/kaggle/input/chest-xray-pneumonia/chest_xray/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'   # or 'categorical' if multi-class
)

val_gen = val_datagen.flow_from_directory(
    "/kaggle/input/chest-xray-pneumonia/chest_xray/test",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')  # binary
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from tensorflow.keras.metrics import Recall

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', Recall()]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=2)
]

In [ ]:
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
 21/163 ━━━━━━━━━━━━━━━━━━━━ 2:13 937ms/step - accuracy: 0.7534 - loss: 1.4274 - recall: 0.8110

In [ ]:
model.fit(train_gen, validation_data=val_gen, epochs=10)

In [ ]:
model.save("cnn_model.h5")


In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("cnn_model")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(model, img_path):
    img = image.load_img(img_path, target_size=(224,224), color_mode='grayscale')

    img_array = image.img_to_array(img)   # shape: (224,224,1)
    img_array = img_array / 255.0

    img_array = np.expand_dims(img_array, axis=0)  # (1,224,224,1)

    pred = model.predict(img_array)[0][0]
    p = model.predict(img_array)

    label = "Pneumonia" if pred > 0.5 else "Normal"
    confidence = pred if pred > 0.5 else 1 - pred

    return label, float(confidence), p

In [ ]:
model.evaluate(val_gen)

In [ ]:
predict_image(model, "/content/NORMAL2-IM-1431-0001.jpeg")

In [ ]:
# New code

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================
# 1. Paths
# =========================
train_dir = "/content/drive/MyDrive/Teaching/PGAPRO02/dl/chest_xray/train"
val_dir = "/content/drive/MyDrive/Teaching/PGAPRO02/dl/chest_xray/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# 2. Data Generators (Grayscale)
# =========================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# =========================
# 3. Model Architecture
# =========================
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

# =========================
# 4. Compile
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 5. Callbacks
# =========================
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2)
]

# =========================
# 6. Train
# =========================
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks
)


In [ ]:
# =========================
# 7. Save Model + Class Map
# =========================

model.save("/content/drive/MyDrive/Teaching/PGAPRO02/dl/cnn_grayscale_model.h5")

import json
with open("/content/drive/MyDrive/Teaching/PGAPRO02/dl/class_map.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("Model saved successfully")

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import json

# =========================
# 1. Load Model
# =========================
model = tf.keras.models.load_model("/content/drive/MyDrive/Teaching/PGAPRO02/dl/cnn_grayscale_model.h5")

# =========================
# 2. Load Class Map
# =========================
with open("/content/drive/MyDrive/Teaching/PGAPRO02/dl/class_map.json") as f:
    class_map = json.load(f)

# reverse mapping
class_map = {v:k for k,v in class_map.items()}

# =========================
# 3. Prediction Function
# =========================
def predict_image(img_path):
    img = image.load_img(
        img_path,
        target_size=(224,224),
        color_mode='grayscale'
    )

    img_array = image.img_to_array(img) / 255.0

    # shape: (224,224,1) → (1,224,224,1)
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array)[0][0]

    class_id = 1 if pred > 0.5 else 0
    label = class_map[class_id]

    return {
        "probability": float(pred),
        "class": label
    }
# =========================
# 4. Test Prediction
# =========================
result = predict_image("/content/person1_bacteria_1.jpeg")
print(result)

In [ ]:
result = predict_image("/content/IM-0127-0001.jpeg")
print(result)

In [ ]:
# Break time : come back by 6:50 pm